In [33]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [34]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


# Task 1: Load and inspect the booking records.

In [35]:
import pandas as pd
import numpy as np
df = pd.read_csv("13_hotel_bookings.csv")
print(df.head())
print("\nshape (rows, coloumns):", df.shape)
print("\ncoloumn info:")
df.info

  Booking_ID Checkin_Date  Lead_Time_Days Room_Type  Room_Rate  Nights  \
0     B00000   2025-05-15             NaN    Deluxe    3080.04       6   
1     B00001   2025-10-14           123.0    Deluxe    6418.87       2   
2     B00002   2026-08-08           103.0  Standard   14073.29       2   
3     B00003   2026-03-20            93.0    Deluxe   14371.81       6   
4     B00004   2025-05-15           159.0  Standard   14898.76       7   

  Booking_Channel Traveler_Type Canceled  
0  Direct Website      Business       No  
1    Travel Agent       Leisure       No  
2    Travel Agent       Leisure       No  
3       Corporate       Leisure      Yes  
4    Travel Agent      Business       No  

shape (rows, coloumns): (761, 9)

coloumn info:


<bound method DataFrame.info of     Booking_ID Checkin_Date  Lead_Time_Days Room_Type  Room_Rate  Nights  \
0       B00000   2025-05-15             NaN    Deluxe    3080.04       6   
1       B00001   2025-10-14           123.0    Deluxe    6418.87       2   
2       B00002   2026-08-08           103.0  Standard   14073.29       2   
3       B00003   2026-03-20            93.0    Deluxe   14371.81       6   
4       B00004   2025-05-15           159.0  Standard   14898.76       7   
..         ...          ...             ...       ...        ...     ...   
756     B00336   2026-02-04            77.0    Deluxe    5430.23       1   
757     B00155   2025-01-28            78.0    Deluxe    9822.49       6   
758     B00528   2025-04-24           148.0     Suite   13356.25       3   
759     B00736   2026-02-09            19.0     Suite    4521.13       5   
760     B00097   2026-02-12            58.0  Standard    9670.63       5   

    Booking_Channel Traveler_Type Canceled  
0    Direc

# Task 2: Identify missing values and duplicate Booking_ID records.

In [36]:
(df .isnull().sum())
(df["Booking_ID"].duplicated().sum())
print("missing values in each column:\n", df.isnull().sum())
print("duplicated values in Booking_ID:\n", df["Booking_ID"].duplicated().sum())

missing values in each column:
 Booking_ID          0
Checkin_Date        0
Lead_Time_Days     18
Room_Type           0
Room_Rate          18
Nights              0
Booking_Channel     0
Traveler_Type       0
Canceled            0
dtype: int64
duplicated values in Booking_ID:
 11


#Task 3: handle missing lead_ time_days or room_rate values using a justified approach.

In [37]:
df = df.drop_duplicates(subset=["Booking_ID"], keep="first")
df["Lead_Time_Days"] = df["Lead_Time_Days"].fillna(df["Lead_Time_Days"].median())
df["Room_Rate"] = df["Room_Rate"].fillna(df["Room_Rate"].median())

print("remaining missing values in each column:\n", df.isnull().sum())
print("\nremaining duplicated values in Booking_ID:\n", df["Booking_ID"].duplicated().sum())
print("Updated shape:", df.shape)

remaining missing values in each column:
 Booking_ID         0
Checkin_Date       0
Lead_Time_Days     0
Room_Type          0
Room_Rate          0
Nights             0
Booking_Channel    0
Traveler_Type      0
Canceled           0
dtype: int64

remaining duplicated values in Booking_ID:
 0
Updated shape: (750, 9)


# Task 4: Convert Checkin_Date to datetime and extract arrival month and weekday.

In [38]:
df["Checkin_Date"] = pd.to_datetime(df["Checkin_Date"])

df["Arrival_Month"] = df["Checkin_Date"].dt.month_name()
df["Arrival_Month_Num"] = df["Checkin_Date"].dt.month
df["Arrival_Weekday"] = df["Checkin_Date"].dt.day_name()

print(df[["Checkin_Date", "Arrival_Month", "Arrival_Month_Num", "Arrival_Weekday"]].head())
print("\nCheckin_Date dtype:", df["Checkin_Date"].dtype)

  Checkin_Date Arrival_Month  Arrival_Month_Num Arrival_Weekday
0   2025-05-15           May                  5        Thursday
1   2025-10-14       October                 10         Tuesday
2   2026-08-08        August                  8        Saturday
3   2026-03-20         March                  3          Friday
4   2025-05-15           May                  5        Thursday

Checkin_Date dtype: datetime64[us]


 Task 5: claculate total bookings, total room nights, and overall cancellation percentage.

In [39]:
total_bookings = len(df)
total_room_nights = df["Nights"].sum()
cancellation_pct = (df["Canceled"] == "Yes").mean() * 100

print("Total bookings:", total_bookings)
print("Total room nights:", total_room_nights)
print("Overall cancellation percentage:", round(cancellation_pct, 2), "%")

Total bookings: 750
Total room nights: 2988
Overall cancellation percentage: 19.87 %


Task 6: Calculate row-wise total booking revenue (Nights *Room_Rate)for non-canceled bookings.

In [40]:
active_df = df[df["Canceled"] != "Yes"].copy()
active_df["Total_Revenue"] = active_df["Nights"] * active_df["Room_Rate"]
df["Total_Revenue"] = np.where(df["Canceled"] != "Yes", df["Nights"] * df["Room_Rate"], 0)

print(active_df[["Booking_ID", "Canceled", "Nights", "Room_Rate", "Total_Revenue"]].head())
print("\nTotal realized revenue (non-canceled):", active_df["Total_Revenue"].sum())
print(df.columns.tolist())

  Booking_ID Canceled  Nights  Room_Rate  Total_Revenue
0     B00000       No       6    3080.04       18480.24
1     B00001       No       2    6418.87       12837.74
2     B00002       No       2   14073.29       28146.58
4     B00004       No       7   14898.76      104291.32
5     B00005       No       2    8455.03       16910.06

Total realized revenue (non-canceled): 21095635.45
['Booking_ID', 'Checkin_Date', 'Lead_Time_Days', 'Room_Type', 'Room_Rate', 'Nights', 'Booking_Channel', 'Traveler_Type', 'Canceled', 'Arrival_Month', 'Arrival_Month_Num', 'Arrival_Weekday', 'Total_Revenue']


Task 7: Compare average room rates and occupancy across room types (e.g., Standrad, Deluxe, Suite). 

In [41]:
room_summary = df.groupby("Room_Type").agg(
    avg_room_rate=("Room_Rate", "mean"),
    total_bookings=("Booking_ID", "count"),
    total_room_nights=("Nights", "sum"),
    total_realized_revenue=("Total_Revenue", "sum"),
).round(2)

print(room_summary)

           avg_room_rate  total_bookings  total_room_nights  \
Room_Type                                                     
Deluxe           8444.09             259               1065   
Standard         8625.93             383               1482   
Suite            9230.19             108                441   

           total_realized_revenue  
Room_Type                          
Deluxe                 7172275.78  
Standard              10665374.52  
Suite                  3257985.14  


#Task 8: Rank booking channels (e.g., Direct Website,OTA,Travel Agent) by booking volume and revenue.


In [42]:
channel_rank = (df.groupby("Booking_Channel")
                .agg(
                    total_volume=("Booking_ID", "count"),
                    total_revenue=("Total_Revenue", "sum"),
                )
                .sort_values("total_revenue", ascending=False)
                .round(2))
print(channel_rank)

                 total_volume  total_revenue
Booking_Channel                             
OTA                       181     5706519.12
Travel Agent              194     5356314.31
Corporate                 197     5217502.47
Direct Website            178     4815299.54


#Task 9: Analyze monthly booking and revenue trends to identify peak months.

In [44]:

monthly_trend = (
    df.groupby("Checkin_Month")
    .agg(
        total_bookings=("Booking_ID", "count"),
        total_revenue=("Total_Revenue", "sum"),  
    )
    .sort_values(by="total_revenue", ascending=False)
    .round(2)
)
print("Columns:", df.columns.tolist())
print(monthly_trend)

KeyError: 'Checkin_Month'